In [1]:
"""
Phase 1: EDA — AI4I 2020 Predictive Maintenance
Goal: understand class balance, sensor behavior under failure, data quality.
"""

'\nPhase 1: EDA — AI4I 2020 Predictive Maintenance\nGoal: understand class balance, sensor behavior under failure, data quality.\n'

# Phase 1 — EDA

Industrial CNC milling-process telemetry. 10K observations, 5 continuous
sensors plus categorical product type. Multi-label failure flags (TWF, HDF,
PWF, OSF, RNF) and a binary `Machine failure` rollup.

**Goals of this notebook:**
1. Confirm dataset shape, dtypes, missingness.
2. Quantify class imbalance — drives the modeling strategy in Phase 3.
3. Profile sensor distributions to understand which signals separate
   failures from healthy runs.
4. Surface multicollinearity, which constrains the linear baseline.
5. Document data-quality issues for the README.

In [2]:
import sys
from pathlib import Path

# Resolve project root whether the notebook runs from notebooks/ or repo root
HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import load

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = load(ROOT / "data" / "ai4i2020.csv")
print(df.shape)
df.head()

(10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M00001,M,300.8,309.1,1649,32.2,130,0,0,0,0,0,0
1,2,L00001,L,300.8,309.1,1783,27.8,48,0,0,0,0,0,0
2,3,H00001,H,300.8,309.2,1586,46.3,165,0,0,0,0,0,0
3,4,M00002,M,300.8,309.2,1384,45.1,188,0,0,0,0,0,0
4,5,L00002,L,300.8,309.2,1836,14.5,107,1,0,0,1,0,0


## Schema & missingness

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [4]:
df.isna().sum()

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

No nulls. Eleven numeric columns, two string (`Product ID`, `Type`). Schema
matches the canonical UCI/Kaggle release.

In [5]:
df.describe().round(2)

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00,10000.0,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,5000.50,300.0,310.00,1541.03,39.82,126.90,0.06,0.01,0.00,0.01,0.03,0.00
std,2886.90,2.0,2.75,176.22,9.95,73.46,0.23,0.08,0.06,0.12,0.18,0.03
min,1.00,297.0,306.10,1168.00,3.80,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,2500.75,298.7,308.00,1418.00,33.20,63.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,5000.50,299.2,309.10,1541.00,39.80,127.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,7500.25,301.4,312.90,1658.00,46.60,190.00,0.00,0.00,0.00,0.00,0.00,0.00
max,10000.00,305.5,316.60,2220.00,76.60,253.00,1.00,1.00,1.00,1.00,1.00,1.00


## Class balance — the central modeling challenge

Failures are rare. This is the headline number that determines everything
downstream: stratified split, recall-oriented metrics, class-imbalance
weighting in the model.

In [6]:
SENSORS = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
]
MODES = ["TWF", "HDF", "PWF", "OSF", "RNF"]

n = len(df)
n_fail = int(df["Machine failure"].sum())
print(f"Total rows           : {n:,}")
print(f"Machine failure = 1  : {n_fail} ({n_fail/n*100:.2f}%)")
print(f"Imbalance ratio      : {(n - n_fail)/n_fail:.1f} : 1 (negative : positive)")

Total rows           : 10,000
Machine failure = 1  : 554 (5.54%)
Imbalance ratio      : 17.1 : 1 (negative : positive)


## Failure-mode breakdown

`Machine failure` is the OR of five mutually-non-exclusive root causes.
Treating this as a single binary problem is fine for a first model, but the
README notes the multi-label extension as future work — different modes have
different sensor signatures and would benefit from per-mode classifiers.

In [7]:
mode_counts = df[MODES].sum().rename("count").to_frame()
mode_counts["rate_%"] = (mode_counts["count"] / n * 100).round(2)
print(mode_counts)

multi_mode = (df[MODES].sum(axis=1) > 1).sum()
print(f"\nRows with >1 active failure mode: {multi_mode}")

# Label consistency: Machine failure should == OR(modes)
inconsistent = (df["Machine failure"] != (df[MODES].sum(axis=1) > 0).astype(int)).sum()
print(f"Rows where Machine_failure != OR(modes): {inconsistent}")

     count  rate_%
TWF     57    0.57
HDF     34    0.34
PWF    144    1.44
OSF    332    3.32
RNF      8    0.08

Rows with >1 active failure mode: 21
Rows where Machine_failure != OR(modes): 0


## Failure rate by product type

L variants fail more often than M or H. This matches Matzka's overstrain
threshold (lower for L, higher for H), and means `Type` is signal — keep it.

In [8]:
by_type = df.groupby("Type")["Machine failure"].agg(["count", "sum", "mean"])
by_type["mean"] = (by_type["mean"] * 100).round(2)
print(by_type.rename(columns={"mean": "fail_rate_%"}))

      count  sum  fail_rate_%
Type                         
H      1954   63         3.22
L      5015  354         7.06
M      3031  137         4.52


## Sensor distributions — failure vs. healthy

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, s in enumerate(SENSORS):
    ax = axes[i]
    sns.kdeplot(data=df, x=s, hue="Machine failure", common_norm=False,
                fill=True, alpha=0.35, ax=ax, palette={0: "#4C78A8", 1: "#E45756"})
    ax.set_title(s)
    ax.set_xlabel("")
axes[-1].axis("off")
fig.suptitle("Sensor distributions — failed (red) vs. healthy (blue)",
             y=1.00, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_sensor_distributions.png", bbox_inches="tight", dpi=130)
plt.close(fig)
print(f"Saved {FIG_DIR / '01_sensor_distributions.png'}")

Saved /home/claude/predictive-maintenance-ai4i/reports/figures/01_sensor_distributions.png


### What the sensor plots show

- **Tool wear** and **Torque** shift right under failure — the strongest
  univariate signals.
- **Rotational speed** shifts left under failure (low rpm + high torque is
  the power-failure signature).
- **Air temp** and **Process temp** look almost identical between groups —
  on their own, marginal. But their *difference* is the cooling signal that
  drives heat-dissipation failures, which is the motivation for the
  `temp_diff` engineered feature in Phase 2.

## Failure-mode prevalence

In [10]:
fig, ax = plt.subplots(figsize=(8, 4))
mode_counts["count"].sort_values().plot(
    kind="barh", ax=ax, color="#54A24B", edgecolor="black"
)
ax.set_xlabel("Count (out of 10,000)")
ax.set_title("Failure modes — prevalence")
for i, v in enumerate(mode_counts["count"].sort_values()):
    ax.text(v + 3, i, str(v), va="center")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_failure_modes.png", bbox_inches="tight", dpi=130)
plt.close(fig)
print(f"Saved {FIG_DIR / '02_failure_modes.png'}")

Saved /home/claude/predictive-maintenance-ai4i/reports/figures/02_failure_modes.png


## Correlation structure

In [11]:
corr = df[SENSORS + ["Machine failure"]].corr()
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Sensor correlation matrix")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_correlations.png", bbox_inches="tight", dpi=130)
plt.close(fig)
print(f"Saved {FIG_DIR / '03_correlations.png'}")

print("\nCorrelation with Machine failure (sorted by |r|):")
print(corr["Machine failure"].drop("Machine failure")
      .reindex(corr["Machine failure"].drop("Machine failure")
               .abs().sort_values(ascending=False).index)
      .round(3))

Saved /home/claude/predictive-maintenance-ai4i/reports/figures/03_correlations.png

Correlation with Machine failure (sorted by |r|):
Tool wear [min]            0.230
Torque [Nm]                0.078
Rotational speed [rpm]    -0.070
Process temperature [K]   -0.008
Air temperature [K]       -0.001
Name: Machine failure, dtype: float64


### Multicollinearity flags

- `Air temp` ↔ `Process temp` ≈ +0.96. Process temp is air temp + ~10 K by
  construction. For the linear baseline this inflates standard errors;
  either use ridge regularization or replace the pair with `temp_diff`. Tree
  models are unaffected.
- `Torque` ↔ `Rotational speed` ≈ -0.94. Mechanical-power conservation
  (P = τ · ω). Same handling: either keep both for trees, or replace with
  the engineered `power` feature for the linear baseline.

## Bivariate view — torque × rotational speed (the PWF surface)

In [12]:
fig, ax = plt.subplots(figsize=(8, 6))
ok = df[df["Machine failure"] == 0]
fail = df[df["Machine failure"] == 1]
ax.scatter(ok["Rotational speed [rpm]"], ok["Torque [Nm]"],
           s=4, alpha=0.18, c="#4C78A8", label="Healthy")
ax.scatter(fail["Rotational speed [rpm]"], fail["Torque [Nm]"],
           s=18, alpha=0.85, c="#E45756", edgecolor="black",
           linewidth=0.3, label="Failure")
ax.set_xlabel("Rotational speed [rpm]")
ax.set_ylabel("Torque [Nm]")
ax.set_title("Failures concentrate at low-rpm / high-torque (PWF + OSF regimes)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "04_torque_rpm_failures.png", bbox_inches="tight", dpi=130)
plt.close(fig)
print(f"Saved {FIG_DIR / '04_torque_rpm_failures.png'}")

Saved /home/claude/predictive-maintenance-ai4i/reports/figures/04_torque_rpm_failures.png


## Tool-wear bands and TWF

In [13]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df.loc[df["Machine failure"] == 0, "Tool wear [min]"],
        bins=40, alpha=0.5, label="Healthy", color="#4C78A8")
ax.hist(df.loc[df["Machine failure"] == 1, "Tool wear [min]"],
        bins=40, alpha=0.7, label="Failure", color="#E45756")
ax.axvspan(200, 240, color="gold", alpha=0.25, label="TWF band [200, 240] min")
ax.set_xlabel("Tool wear [min]")
ax.set_ylabel("Count")
ax.set_title("Failures cluster at high tool wear; TWF band highlighted")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "05_tool_wear.png", bbox_inches="tight", dpi=130)
plt.close(fig)
print(f"Saved {FIG_DIR / '05_tool_wear.png'}")

Saved /home/claude/predictive-maintenance-ai4i/reports/figures/05_tool_wear.png


## Data-quality summary

| Issue | Severity | Handling |
|---|---|---|
| Severe class imbalance (~3-4% positive) | High | Stratified split, `scale_pos_weight`, recall/F1/PR-AUC |
| Strong multicollinearity (air↔process temp, torque↔rpm) | Medium | Engineered features (temp_diff, power); ridge for linear baseline |
| RNF (random failure, 0.1%) is by design unpredictable | Medium | Optionally exclude RNF-only rows from training; documented as irreducible noise |
| Per-mode failure flags would leak the label | Critical | TWF/HDF/PWF/OSF/RNF must be dropped from features (kept only for diagnostics) |
| UDI and Product ID are row identifiers | Low | Drop from features; not predictive |
| No nulls, no duplicate UDIs, schema clean | Positive | No imputation needed |
print("Phase 1 complete — figures saved to reports/figures/")